# 🧪 Approach 1: Classical Computer Vision Feature Engineering & Machine Learning

**Objective:** Implement classical handcrafted feature descriptors (CIELAB color space, HSV histograms, Local Binary Patterns for texture roughness, GLCM descriptors) and train Support Vector Machine (SVM) and Random Forest classifiers.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from src.dataset import prepare_dataset_splits
from src.classical_model import extract_features_batch, ClassicalClassifierPipeline

plt.style.use('seaborn-v0_8-whitegrid')


## 1. Load Dataset Splits and Feature Extraction Pipeline
Extract 128-dimensional color and texture feature vectors for each sample.


In [ ]:
splits = prepare_dataset_splits('.')
classes = splits['classes']
tr_paths, tr_labels = splits['train']
te_paths, te_labels = splits['test']

print('Extracting features for Training and Testing sets...')
X_train = extract_features_batch(tr_paths)
y_train = np.array(tr_labels)

X_test = extract_features_batch(te_paths)
y_test = np.array(te_labels)

print(f"Feature matrix X_train shape: {X_train.shape}")
print(f"Feature matrix X_test shape:  {X_test.shape}")


## 2. Train and Evaluate Support Vector Machine (RBF Kernel)


In [ ]:
svm_pipeline = ClassicalClassifierPipeline(model_type='svm')
svm_pipeline.fit(X_train, y_train, class_names=classes)
svm_res = svm_pipeline.evaluate(X_test, y_test)

print(f"SVM Test Accuracy: {svm_res['accuracy']*100:.2f}%")
print(f"SVM Macro F1-Score: {svm_res['macro_f1']:.4f}")
print("\nClassification Report:")
print(pd.DataFrame(svm_res['classification_report']).T)


## 3. Train and Evaluate Random Forest Classifier (200 Trees)


In [ ]:
rf_pipeline = ClassicalClassifierPipeline(model_type='rf')
rf_pipeline.fit(X_train, y_train, class_names=classes)
rf_res = rf_pipeline.evaluate(X_test, y_test)

print(f"Random Forest Test Accuracy: {rf_res['accuracy']*100:.2f}%")
print(f"Random Forest Macro F1-Score: {rf_res['macro_f1']:.4f}")
print("\nClassification Report:")
print(pd.DataFrame(rf_res['classification_report']).T)


## 4. Confusion Matrix Comparison (SVM vs RF)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(svm_res['confusion_matrix'], annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=axes[0])
axes[0].set_title('Classical SVM Confusion Matrix', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Ground Truth')

sns.heatmap(rf_res['confusion_matrix'], annot=True, fmt='d', cmap='Greens', xticklabels=classes, yticklabels=classes, ax=axes[1])
axes[1].set_title('Random Forest Confusion Matrix', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Ground Truth')
plt.tight_layout()
plt.show()
